# Módulo 3 · ANN: Recomendación de acción preventiva


In [30]:
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from tensorflow import keras


In [31]:
df = pd.read_csv("../data/raw/vectoria_acciones.csv")


In [32]:
df["ratio_casos"] = df["casos_pred_4sem"] / df["casos_umbral_endemico"]


In [33]:
 #Variables de entrada (X) y etiqueta (y) ----------
COLS = ["casos_pred_4sem", "casos_umbral_endemico", "ratio_casos",
        "lluvia_14d_mm", "temp_max", "aegypti_detectado"]
X = df[COLS].values

In [34]:
# La red solo trabaja con números: convertimos las 4 acciones a 0, 1, 2, 3
le = LabelEncoder().fit(df["recomendacion"])
y = le.transform(df["recomendacion"])

In [35]:
# Cada semana aparece 5 veces (5 escenarios). Usamos la fecha de inicio #como identificador de grupo para que una semana nunca quede repartida #entre entrenamiento y prueba
grupos = df["fecha_inicio_semana"]
print("Filas:", len(df), "| Semanas distintas:", grupos.nunique(), "(debe ser 412)")

Filas: 2060 | Semanas distintas: 412 (debe ser 412)


In [36]:
#Aqui entrenamos la red
def entrenar(X_train, y_train, semilla):
    """Entrena una ANN y la devuelve.
    La semilla hace los resultados repetibles: misma semilla, mismo resultado."""
    tf.keras.utils.set_random_seed(semilla)
    modelo = keras.Sequential([
        keras.layers.Input(shape=(X_train.shape[1],)),        # una entrada por variable
        keras.layers.Dense(64, activation="relu"),            # capa oculta 1
        keras.layers.Dense(32, activation="relu"),            # capa oculta 2
        keras.layers.Dense(len(le.classes_), activation="softmax"),  # una salida por acción
    ])
    # adam = optimizador; la pérdida sparse_categorical_crossentropy sirve para
    # clasificación multiclase con etiquetas numéricas
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    # 150 pasadas por los datos, de 32 filas en 32
    modelo.fit(X_train, y_train, epochs=150, batch_size=32, verbose=0)
    return modelo

EVALUACIÓN: validación cruzada de 5 particiones, agrupada por semana
Las 412 semanas se reparten en 5 bloques. En cada vuelta, 4 bloques
entrenan y 1 bloque prueba. Así CADA semana se predice una vez con un
modelo que nunca la vio, y las métricas salen de las 2,060 filas

In [47]:
pred_una = np.zeros(len(df), dtype=int)     # predicciones de una sola red
pred_ens = np.zeros(len(df), dtype=int)     # predicciones de 3 redes

for k, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=grupos)):
    sc = StandardScaler().fit(X[tr])
    X_tr, X_te = sc.transform(X[tr]), sc.transform(X[te])

    # Entrenamos 3 redes con semillas distintas; cada una devuelve 4 probabilidades por fila
    probs = [entrenar(X_tr, y[tr], semilla).predict(X_te, verbose=0)
             for semilla in (1, 2, 3)]

    pred_una[te] = probs[0].argmax(axis=1)                  # una sola red: la clase más probable
    pred_ens[te] = np.mean(probs, axis=0).argmax(axis=1)    # ensamble: promedio de las 3 redes
    print(f"Bloque {k + 1}/5 listo")

print("\n UNA RED (validación cruzada, 2060 filas)")
print(classification_report(y, pred_una, target_names=le.classes_))

print(" 3 REDES (validación cruzada, 2060 filas)")
print(classification_report(y, pred_ens, target_names=le.classes_))

Bloque 1/5 listo
Bloque 2/5 listo
Bloque 3/5 listo
Bloque 4/5 listo
Bloque 5/5 listo

=== UNA RED (validación cruzada, 2060 filas) ===
                       precision    recall  f1-score   support

     alerta_sanitaria       0.95      0.92      0.94       379
eliminacion_criaderos       0.90      0.85      0.88       321
fumigacion_focalizada       0.91      0.85      0.88       147
     monitoreo_normal       0.95      0.98      0.96      1213

             accuracy                           0.94      2060
            macro avg       0.93      0.90      0.91      2060
         weighted avg       0.94      0.94      0.94      2060

=== ENSAMBLE DE 3 REDES (validación cruzada, 2060 filas) ===
                       precision    recall  f1-score   support

     alerta_sanitaria       0.97      0.92      0.94       379
eliminacion_criaderos       0.88      0.88      0.88       321
fumigacion_focalizada       0.90      0.85      0.87       147
     monitoreo_normal       0.95      0.98  

MODELO FINAL: una sola red, entrenada con TODOS los datos
La validación cruzada ya midió el rendimiento real; aquí usamos todas las
semanas para que el modelo final aprenda lo más posible.

In [49]:
scaler = StandardScaler().fit(X)            # escalador con todos los datos
X_s = scaler.transform(X)

Path("../models").mkdir(exist_ok=True)      # crea /models si no existe
entrenar(X_s, y, 1).save("../models/modelo_ann_acciones.keras")

# Guardamos cómo preparar datos nuevos: escalador, codificador de etiquetas
joblib.dump({"scaler": scaler, "label_encoder": le, "features": COLS},
            "../models/modelo_ann_acciones_preprocesadores.pkl")
print("Guardado en models/: modelo_ann_acciones.keras + preprocesadores.pkl")

Guardado en models/: modelo_ann_acciones.keras + preprocesadores.pkl
